# DAPO/GRPO with Nemotron Nano 3.5 and NeMo Gym

This notebook prepares Gym-compatible DAPO/AIME JSONL and runs one end-to-end optimizer step. Complete the public setup in the parent `README.md` and export `SHARED_ROOT` before starting Jupyter.

In [ ]:
import os
from pathlib import Path

shared_value = os.environ.get('SHARED_ROOT')
if not shared_value:
    raise RuntimeError('Export SHARED_ROOT before starting Jupyter.')

SHARED_ROOT = Path(shared_value).expanduser().resolve()
NEMOTRON_REPO = Path(os.environ.get('NEMOTRON_REPO', SHARED_ROOT / 'code/Nemotron')).expanduser().resolve()
NEMO_RL_IMAGE = os.environ.get('NEMO_RL_IMAGE', 'nemo-rl:nemotron-nano-3.5')
ASSETS = NEMOTRON_REPO / 'usage-cookbook/Nemotron-Nano-3.5/RL/grpo-dapo-nemo-gym'
MODEL = SHARED_ROOT / 'models/nemotron-nano-3.5-ea2'

try:
    assets_relative = ASSETS.relative_to(SHARED_ROOT)
except ValueError as exc:
    raise RuntimeError('NEMOTRON_REPO must be located under SHARED_ROOT.') from exc

ASSETS_CONTAINER = Path('/shared') / assets_relative
RECIPE_CONTAINER = ASSETS_CONTAINER / 'dapo_nano_3_5_starter_nemo_gym.yaml'
PREPARE_CONTAINER = ASSETS_CONTAINER / 'prepare_hf_dapo_data_for_nemo_gym.py'
assert (ASSETS / 'dapo_nano_3_5_starter_nemo_gym.yaml').is_file()
assert (ASSETS / 'prepare_hf_dapo_data_for_nemo_gym.py').is_file()
assert (MODEL / 'config.json').is_file(), MODEL

os.environ['SHARED_ROOT'] = str(SHARED_ROOT)
os.environ['NEMO_RL_IMAGE'] = NEMO_RL_IMAGE
os.environ['RECIPE_CONTAINER'] = str(RECIPE_CONTAINER)
os.environ['PREPARE_CONTAINER'] = str(PREPARE_CONTAINER)
print(f'Assets: {ASSETS}')
print(f'Model: {MODEL}')
print(f'Image: {NEMO_RL_IMAGE}')

## Prepare smoke and validation data

The converter adds the `math_with_judge_simple_agent` routing field required by NeMo Gym.

In [ ]:
%%bash
set -euo pipefail
mkdir -p "$SHARED_ROOT/.cache/huggingface" "$SHARED_ROOT/data/dapo_nano_3_5_nemo_gym"

docker run --rm \
  -e HF_HOME=/shared/.cache/huggingface -e HF_TOKEN \
  -v "$SHARED_ROOT:/shared" -w /opt/nemo-rl "$NEMO_RL_IMAGE" \
  /opt/nemo_rl_venv/bin/python "$PREPARE_CONTAINER" \
  --dataset BytedTsinghua-SIA/DAPO-Math-17k \
  --cache-dir /shared/.cache/huggingface --limit 4 \
  --output /shared/data/dapo_nano_3_5_nemo_gym/smoke_train.jsonl

docker run --rm \
  -e HF_HOME=/shared/.cache/huggingface -e HF_TOKEN \
  -v "$SHARED_ROOT:/shared" -w /opt/nemo-rl "$NEMO_RL_IMAGE" \
  /opt/nemo_rl_venv/bin/python "$PREPARE_CONTAINER" \
  --dataset BytedTsinghua-SIA/AIME-2024 \
  --cache-dir /shared/.cache/huggingface \
  --output /shared/data/dapo_nano_3_5_nemo_gym/validation.jsonl

In [ ]:
import json

smoke_path = SHARED_ROOT / 'data/dapo_nano_3_5_nemo_gym/smoke_train.jsonl'
row = json.loads(smoke_path.read_text().splitlines()[0])
assert row['agent_ref']['name'] == 'math_with_judge_simple_agent'
assert {'responses_create_params', 'question', 'expected_answer'} <= row.keys()
row

## Run one NeMo Gym optimizer step

The workload starts the Gym services and vLLM Responses API, collects four rollouts, verifies rewards, computes log probabilities, and updates the policy.

In [ ]:
%%bash
set -euo pipefail
mkdir -p \
  "$SHARED_ROOT/logs/dapo_nano_3_5_starter_nemo_gym" \
  "$SHARED_ROOT/results/dapo_nano_3_5_starter_nemo_gym"

docker run --rm --gpus all --ipc=host --network=host \
  --ulimit memlock=-1 --ulimit stack=67108864 \
  -e CUDA_VISIBLE_DEVICES=0,1,2,3 \
  -e HF_HOME=/shared/.cache/huggingface \
  -e HF_TOKEN \
  -v "$SHARED_ROOT:/shared" -w /opt/nemo-rl "$NEMO_RL_IMAGE" \
  /opt/nemo_rl_venv/bin/python examples/nemo_gym/run_grpo_nemo_gym.py \
  --config "$RECIPE_CONTAINER" \
  grpo.num_prompts_per_step=1 \
  grpo.num_generations_per_prompt=4 \
  grpo.max_num_steps=1 \
  grpo.val_period=-1 \
  grpo.val_at_start=false \
  grpo.val_at_end=false \
  policy.train_global_batch_size=4 \
  policy.max_total_sequence_length=1024 \
  policy.generation.max_new_tokens=256 \
  policy.generation.vllm_cfg.max_model_len=1024 \
  data.train.data_path=/shared/data/dapo_nano_3_5_nemo_gym/smoke_train.jsonl \
  logger.tensorboard_enabled=false \
  logger.monitor_gpus=false \
  checkpointing.enabled=false

A successful run completes all four Gym rollouts and one policy update. Smoke-test rewards and losses are wiring checks, not model-quality measurements.